In [24]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field
import operator
import os

In [25]:
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

generator_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

evaluator_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

optimizer_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

In [26]:
class TweetState(TypedDict):

    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvements"]
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [27]:
class evaluationSchema(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(description='evaluate the tweet')
    feedback: str = Field(description='give a feedback')

In [28]:
strucuted_evaluator_model = evaluator_model.with_structured_output(evaluationSchema)

In [29]:
def generate(state: TweetState):
    messages = [
            SystemMessage(content="You are a funny and clever Twitter/X influencer."),
            HumanMessage(content=f"""
    Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

    Rules:
    - Do NOT use question-answer format.
    - Max 280 characters.
    - Use observational humor, irony, sarcasm, or cultural references.
    - Think in meme logic, punchlines, or relatable takes.
    - Use simple, day to day english
    """)
        ]
    
    response = generator_model.invoke(messages)

    return {'tweet' : response.content, 'tweet_history' : [response.content]}

In [30]:
def evaluate(state: TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
    Evaluate the following tweet:

    Tweet: "{state['tweet']}"

    Use the criteria below to evaluate the tweet:

    1. Originality – Is this fresh, or have you seen it a hundred times before?  
    2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
    3. Punchiness – Is it short, sharp, and scroll-stopping?  
    4. Virality Potential – Would people retweet or share it?  
    5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

    Auto-reject if:
    - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
    - It exceeds 280 characters
    - It reads like a traditional setup-punchline joke
    - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

    ### Respond ONLY in structured format:
    - evaluation: "approved" or "needs_improvement"  
    - feedback: One paragraph explaining the strengths and weaknesses 
    """)
    ]

    response = strucuted_evaluator_model.invoke(messages)
    return {'evaluation' : response.evaluation, 'feedback' : response.feedback, 'feedback_history': [response.feedback]}

In [31]:
def optimize(state: TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
        Improve the tweet based on this feedback:
        "{state['feedback']}"

        Topic: "{state['topic']}"
        Original Tweet:
        {state['tweet']}

        Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
        """)
            ]
    
    iteration = state['iteration'] + 1
    
    response = optimizer_model.invoke(messages).content
    return {'tweet' : response, 'tweet_history': [response], 'iteration': iteration}

In [32]:
def route_evaluation(state: TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [33]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate)
graph.add_node('evaluate', evaluate)
graph.add_node('optimize', optimize)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved' : END, 'needs_improvement' : 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [38]:
initial_state = {
    "topic": "AI",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

In [39]:
result

{'topic': 'AI',
 'tweet': "AI: the only coworker who never steals your lunch, but will steal your job, your memes, and your soul. Meanwhile, it still can't figure out why humans love pineapple on pizza. #FutureIsNow",
 'evaluation': 'approved',
 'feedback': 'The tweet blends familiar AI‑anxiety tropes with fresh, funny specifics (the lunch‑stealing coworker, meme‑theft, and the pineapple‑on‑pizza jab), delivering a witty, relatable punch without feeling stale. It’s concise enough for the scroll‑stopper zone, stays under 280 characters, and ends on a strong, meme‑ready line rather than a weak filler. The hashtag adds shareability, making it ripe for retweets. Overall it scores well on originality, humor, punchiness, virality, and format.',
 'iteration': 1,
 'max_iteration': 5,
 'tweet_history': ["AI: the only coworker who never steals your lunch, but will steal your job, your memes, and your soul. Meanwhile, it still can't figure out why humans love pineapple on pizza. #FutureIsNow"],
 